# Deteccion de oportunidades (outliers espaciales)

Este notebook busca **candidatos a oportunidades** en avisos de venta en CABA a partir de **residuos** de modelos espaciales (precio observado vs precio esperado).

- Un residuo negativo significa: el aviso esta **mas barato de lo que el modelo espera** dado su ubicacion y caracteristicas.
- Un residuo positivo significa: el aviso esta **mas caro de lo que el modelo espera**.

> Importante: un outlier no siempre es una oportunidad. Puede ser error de datos, geocodificacion incorrecta, datos incompletos o un caso especial.


## Setup

- Se cargan dependencias y clases del proyecto (modelos y detectores).
- El `sys.path` se ajusta para importar desde el repo (esto es comodo en notebook, pero en produccion convendria un paquete instalable).


In [1]:
import folium
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from skgstat import Variogram
import geopandas as gpd
from shapely.geometry import Point
import statsmodels.api as sm
import ast
from shapely.ops import unary_union
from pykrige.ok import OrdinaryKriging
from esda import G_Local
import matplotlib.pyplot as plt
from libpysal.weights import Kernel
import os
import sys
PROJECT_ROOT = os.path.abspath("..")  
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
sys.path.append(os.path.abspath("../models"))
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from esda.moran import Moran_Local
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from patsy import dmatrices
from pykrige.rk import RegressionKriging
from statsmodels.robust.scale import mad
from libpysal.weights import Kernel
from models.GWRModel import GWRModel
from models.RFRKModel import RegressionKrigingModel
from models.SARModel import SpatialAutoregressiveModel
from models.modelEvaluator import ModelEvaluator
from models.spatialOutlierDetector import SpatialOutlierDetector
from models.strategies import *



## Datos

- Fuente: `../storage/data/arg_venta_caba_processed.csv`.
- Se construye un `GeoDataFrame` con `latitud/longitud` en **EPSG:4326**.
- Se cargan limites de barrios desde `../barrios.geojson` (util para mapas/joins posteriores).

Nota: mas adelante se reproyecta a **EPSG:3857** para que las distancias (metros) tengan sentido en metodos espaciales basados en vecinos.


In [2]:
df = pd.read_csv("../storage/data/arg_venta_caba_processed.csv")

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitud, df.latitud),
    crs="EPSG:4326"
)
barrios = gpd.read_file("../barrios.geojson")


## Metodologia

1. Se entrenan modelos espaciales sobre el dataset completo.
2. Se calculan **residuos**: `residuo = y - y_pred` (aca `y` es `log_precio`).
3. Se detectan outliers usando estrategias sobre los residuos.

### Estrategias implementadas

- `negative`: lista todos los registros con residuo < 0, ordenados de mas negativo a menos negativo.
- `ztest`: outliers por z-score (global o local si se provee matriz de pesos espaciales). Puede ser robusto (mediana/MAD).
- `quantile`: outliers por cuantiles globales (ej: 5% mas bajo y 5% mas alto).
- `lisa`: outliers espaciales via **Local Moran's I (LISA)** sobre los residuos. En este notebook se guarda el cuadrante `LH` (bajo rodeado de alto), que suele ser el caso mas interesante como anomalia local.

Salida: cada estrategia escribe un CSV en `output/venta/<Modelo>/...`.


In [3]:

STRATEGIES = {
    "negative": NegativeResidualsStrategy(),
    "ztest": ZTestStrategy(),
    "quantile": QuantileStrategy(),
    "lisa": LISAStrategy()
}
def detect_model_outliers(
    *,
    model,
    residuals,
    gdf,
    coords,
    output_dir,
    methods=None,
    params_for_methods=None,
    k_neighbors=15,
    **legacy_params_for_methods
):

    os.makedirs(output_dir, exist_ok=True)

    params_for_methods = params_for_methods or {}
    if legacy_params_for_methods:
        params_for_methods = {**params_for_methods, **legacy_params_for_methods}

    if methods is None:
        methods = ["negative", "ztest", "quantile", "lisa"]

    model_name = model.__class__.__name__
    res = residuals
    coords = model.coords if hasattr(model, "coords") else coords

    detector = SpatialOutlierDetector()

    results = {}

    w = None
    if any(m in methods for m in ["ztest", "lisa"]):

        w = Kernel(
            coords,
            k=k_neighbors,
            function="gaussian",
            fixed=False
        )

    for method in methods:

        strategy = STRATEGIES[method]
        params_for_method = params_for_methods.get(method, {})

        result = strategy.run(
            res=res,
            gdf=gdf,
            coords=coords,
            detector=detector,
            w=w,
            params_for_method=params_for_method,
            output_dir=output_dir,
            model_name=model_name
        )

        results[method] = result

    return results

def run_outlier_analysis_for_models(
    *,
    models: list,
    coords,
    gdf,
    features,
    response,
    output_dir: str,
    methods=['lisa'],
    params_for_methods=None
):
    """
    Corre detección de outliers para múltiples modelos.

    Parameters
    ----------
    models : list
        Lista de instancias de modelos ya entrenados.
    X, y, coords : datos
    gdf : GeoDataFrame original
    output_dir : carpeta base para guardar resultados
    """

    all_results = {}
    residuals_models = {}

    # -------------------------------------------------
    # 1️⃣ Calcular residuos para cada modelo
    # -------------------------------------------------
    for i in range(len(models)):
        model = models[i]
        f = features[i]
        
        model_name = model.__class__.__name__

        X = gdf[f]
        y = gdf[response].to_numpy().reshape(-1,1)

        print(f"Calculando residuos para {model_name}...")

        y_pred = model.in_sample_predictions()
        y = np.asarray(y).ravel()
        y_pred = np.asarray(y_pred).ravel()

        residuals = y - y_pred

        
        residuals = y - y_pred

        residuals_models[model_name] = residuals

    # -------------------------------------------------
    # 2️⃣ Detectar outliers para cada modelo
    # -------------------------------------------------
    for i in range(len(models)):

        model_name = models[i].__class__.__name__
        print(f"Detectando outliers para {model_name}...")

        results = detect_model_outliers(
            model=models[i],
            residuals=residuals_models[model_name],
            gdf=gdf,
            coords=coords,
            output_dir=f"{output_dir}/{model_name}",
            methods=methods,
            params_for_methods=params_for_methods
        )

        all_results[model_name] = results

    return all_results

def _fresh_model_like(model):
    """Crea una instancia nueva con los mismos hiperparámetros básicos.
    Evita copiar estado fitted (mgwr/spreg) para poder re-entrenar en CV.
    """
    cls = model.__class__
    name = cls.__name__

    if name == "GWRModel":
        return cls(gwr_params=getattr(model, "gwr_params", None))
    if name == "RegressionKrigingModel":
        return cls(
            rf_params=getattr(model, "rf_params", None),
            kriging_params=getattr(model, "kriging_params", None),
        )
    if name == "SpatialAutoregressiveModel":
        return cls(k=getattr(model, "k", 5))

    return cls()

def compute_oof_residuals_for_models(
    *,
    models: list,
    coords,
    gdf,
    features,
    response,
    split_mode="geo",  # "geo" o "random"
    geo_split_col="comuna",  # o "barrio"
    n_splits=5,
    random_state=42,
    fit_kwargs_by_model=None,
    fit_kwargs=None,
):
    """Calcula residuos out-of-fold (OOF) por modelo.
    Cada observación se predice una vez en un fold donde no fue entrenada.
    """
    from sklearn.model_selection import KFold, GroupKFold

    fit_kwargs_by_model = fit_kwargs_by_model or {}
    fit_kwargs = fit_kwargs or {}

    coords = np.asarray(coords)
    if coords.ndim != 2 or coords.shape[1] != 2:
        raise ValueError(f"`coords` debe ser (n, 2). Recibido {coords.shape}.")

    y_true = np.asarray(gdf[response].to_numpy()).reshape(-1)
    n = len(y_true)

    groups = None
    if split_mode == "geo" and geo_split_col is not None and geo_split_col in gdf.columns:
        groups = gdf[geo_split_col].fillna("__NA__").astype(str).to_numpy()
        n_groups = len(np.unique(groups))
        n_splits_eff = min(int(n_splits), n_groups)
        if n_splits_eff >= 2:
            splitter = GroupKFold(n_splits=n_splits_eff)
        else:
            splitter = KFold(n_splits=min(int(n_splits), n), shuffle=True, random_state=random_state)
            groups = None
    else:
        splitter = KFold(n_splits=min(int(n_splits), n), shuffle=True, random_state=random_state)

    residuals_models = {}
    preds_models = {}

    for i in range(len(models)):
        base_model = models[i]
        model_name = base_model.__class__.__name__
        f = features[i]

        X_all = gdf[f]
        y_all = np.asarray(gdf[response].to_numpy()).reshape(-1, 1)

        oof_pred = np.full(n, np.nan, dtype=float)

        for fold, (train_idx, val_idx) in enumerate(splitter.split(X_all, y_true, groups)):
            print(f"[{model_name}] Fold {fold+1}: train={len(train_idx)} val={len(val_idx)}")
            model_fold = _fresh_model_like(base_model)
            model_fit_kwargs = dict(fit_kwargs)
            model_fit_kwargs.update(fit_kwargs_by_model.get(model_name, {}))
            model_fold.fit(
                X_all.iloc[train_idx],
                y_all[train_idx],
                coords[train_idx],
                **model_fit_kwargs,
            )
            y_pred_val = model_fold.predict(X_all.iloc[val_idx], coords[val_idx])
            oof_pred[val_idx] = np.asarray(y_pred_val).reshape(-1)

        if np.isnan(oof_pred).any():
            missing = int(np.isnan(oof_pred).sum())
            raise RuntimeError(f"OOF incompleto para {model_name}: faltan {missing} predicciones.")

        preds_models[model_name] = oof_pred
        residuals_models[model_name] = y_true - oof_pred

    return residuals_models, preds_models

def run_oof_outlier_analysis_for_models(
    *,
    models: list,
    coords,
    gdf,
    features,
    response,
    output_dir: str,
    methods=['lisa'],
    params_for_methods=None,
    split_mode="geo",
    geo_split_col="comuna",
    n_splits=5,
    random_state=42,
    fit_kwargs_by_model=None,
    fit_kwargs=None,
):
    """Corre detección de outliers usando residuos OOF (out-of-fold)."""

    os.makedirs(output_dir, exist_ok=True)

    residuals_models, _ = compute_oof_residuals_for_models(
        models=models,
        coords=coords,
        gdf=gdf,
        features=features,
        response=response,
        split_mode=split_mode,
        geo_split_col=geo_split_col,
        n_splits=n_splits,
        random_state=random_state,
        fit_kwargs_by_model=fit_kwargs_by_model,
        fit_kwargs=fit_kwargs,
    )

    all_results = {}
    for i in range(len(models)):
        model_name = models[i].__class__.__name__
        print(f"Detectando outliers OOF para {model_name}...")

        results = detect_model_outliers(
            model=models[i],
            residuals=residuals_models[model_name],
            gdf=gdf,
            coords=coords,
            output_dir=f"{output_dir}/{model_name}",
            methods=methods,
            params_for_methods=params_for_methods,
        )
        all_results[model_name] = results

    return all_results


## Entrenamiento de modelos

Se entrenan tres enfoques (con diferentes sets de features):

- `GWRModel` (Geographically Weighted Regression)
- `RegressionKrigingModel` (Random Forest + Kriging de residuos)
- `SpatialAutoregressiveModel` (SAR)

Para construir vecinos espaciales se reproyecta el `GeoDataFrame` a **EPSG:3857** y se arma `coords` con `x/y` en metros.


In [4]:
# Modelos a comparar (se entrenan 1 vez y luego se analizan sus residuos)
GWRmodel = GWRModel()
RFRKmodel = RegressionKrigingModel()
SARmodel = SpatialAutoregressiveModel()

# Features por modelo (pueden variar segun lo que el modelo soporte y la calidad de la data)
features_gwr = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'antiguedad',
    'expensas',
    'estado_num'
]

features_rfrk = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'antiguedad',
    'expensas',
    'banos',
    'cocheras',
    'estado_num'
]

features_SAR = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'antiguedad',
    'expensas',
    'banos',
    'cocheras',
    'estado_num'
]

features_for_models = [features_gwr, features_rfrk, features_SAR]
response = 'log_precio'
models = [GWRmodel, RFRKmodel, SARmodel]

# Reproyeccion a metros: necesario para vecinos / kernels basados en distancia
gdf = gdf.to_crs(epsg=3857)
coords_tot = np.column_stack([gdf.geometry.x, gdf.geometry.y])

# Parametros de fit por modelo (valores iniciales; ajustalos a gusto)
FIT_PARAMS_BY_MODEL = {
    "SpatialAutoregressiveModel": {"k": 50},
    "RegressionKrigingModel": {"n_closest_points": 105, "variogram_model": "exponential"},
    "GWRModel": {"bw": 250},
}

for i in range(len(models)):
    model = models[i]
    features = features_for_models[i]

    X_tot = gdf[features]
    Y_tot = gdf[response].to_numpy().reshape(-1, 1)

    model_name = model.__class__.__name__
    if model_name not in FIT_PARAMS_BY_MODEL:
        raise KeyError(f"Faltan parametros de fit para {model_name}. Definilos en FIT_PARAMS_BY_MODEL.")
    model.fit(X_tot, Y_tot, coords_tot, **FIT_PARAMS_BY_MODEL[model_name])


KeyboardInterrupt: 

## Ejecucion y lectura de resultados

`run_outlier_analysis_for_models(...)` corre la deteccion y guarda CSVs por modelo. En este notebook el default es `methods=['lisa']` (se puede ampliar a `['negative','ztest','quantile','lisa']`).

Sugerencias para el analisis:

- Priorizar candidatos con residuo muy negativo (en log-precio, un residuo de -0.20 es aprox -18% en precio: `exp(-0.20)-1`).
- Validar manualmente: ubicacion (pin en mapa), superficie/ambientes coherentes, expensas, y si el aviso parece real.


In [ ]:
gdf = gdf.reset_index(drop=True)
coords = np.array([(geom.x, geom.y) for geom in gdf.geometry])


response = 'log_precio'

models = [GWRmodel, RFRKmodel, SARmodel]
params_for_metods = {
    "ztest": {"robust": True, },
    "lisa": {},
    "quantile": {"lower_q": 0.05, "upper_q": 1}
}
metrics = {}
residuals_models = {}


OUTPUT_BASE_DIR = "output/venta"

# ------------------------
# Outliers con residuos in-sample
# ------------------------
results_insample = run_outlier_analysis_for_models(
    models=models,
    coords=coords,
    gdf=gdf,
    methods=["ztest", "quantile", "lisa"],
    params_for_methods=params_for_metods,
    features=features_for_models,
    response=response,
    output_dir=f"{OUTPUT_BASE_DIR}/insample",
)

# ------------------------
# Outliers con residuos OOF (out-of-sample / out-of-fold)
# ------------------------
results_oof = run_oof_outlier_analysis_for_models(
    models=models,
    coords=coords,
    gdf=gdf,
    methods=["ztest", "quantile", "lisa"],
    params_for_methods=params_for_metods,
    features=features_for_models,
    response=response,
    output_dir=f"{OUTPUT_BASE_DIR}/oof",
    split_mode="geo",
    geo_split_col="comuna",
    n_splits=5,
    random_state=42,
    fit_kwargs_by_model=FIT_PARAMS_BY_MODEL,
)

Calculando residuos para GWRModel...
Calculando residuos para RegressionKrigingModel...
Calculando residuos para SpatialAutoregressiveModel...
Detectando outliers para GWRModel...
Detectando outliers para RegressionKrigingModel...
Detectando outliers para SpatialAutoregressiveModel...
[GWRModel] Fold 1: train=5860 val=1473
Model type                                                         Gaussian
Number of observations:                                                5860
Number of covariates:                                                     7

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                           1723.887
Log-likelihood:                                                   -4729.925
AIC:                                                               9473.850
AICc:                                                              9475.874
BIC:                                    